In [1]:
!pip install python-dotenv huggingface-hub llama-index transformers sentence-transformers llama-index-llms-huggingface llama-index-embeddings-huggingface pdfplumber llama-index-llms-openrouter llama-index-retrievers-bm25 tabula-py  jpype1 pystemmer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.3/669.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppresses TensorFlow warnings
from dotenv import load_dotenv
import Stemmer
import nest_asyncio
import tabula
import pandas as pd
from dotenv import load_dotenv
from llama_index.core import Document
from llama_index.core import (SimpleDirectoryReader, VectorStoreIndex, Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
import asyncio


load_dotenv()  # Load environment variables from .env file
# ✅ Load API Key Securely (No Hardcoding!)
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    print("✅ API Key Loaded Successfully:", api_key[:5] + "..." + api_key[-5:])
else:
    print("⚠️ API Key is missing! Check your .env file.")


# ✅ Initialize OpenRouter LLM
llm = OpenRouter(api_key=api_key, model="mistralai/mistral-7b-instruct", max_tokens=512, context_window=4096)
Judge_llm = OpenRouter(api_key=api_key, model="qwen/qwen-turbo", max_tokens=512, context_window=4096)
Settings.llm = llm

# ✅ Apply nest_asyncio to fix event loop issues in Jupyter
nest_asyncio.apply()

# ✅ Set up embedding model - use LegalBERT
embed_model_name = "nlpaueb/legal-bert-base-uncased"
embed_model = HuggingFaceEmbedding(model_name=embed_model_name)
Settings.embed_model = embed_model

✅ API Key Loaded Successfully: sk-or...614bf


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
!pip install pymupdf
!pip install pytesseract

In [7]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import os
import re

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

# ====== PDF file paths ======
pdf_paths = ["law1.pdf", "law2.pdf"]  # Update with your actual filenames

# ====== Text cleaning ======
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\u2022•]", "", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])([A-Za-z])", r"\1 \2", text)
    return text.strip()

# ====== Load and process PDFs with optional OCR ======
documents = []

for path in pdf_paths:
    print(f"\n Processing file: {path}")
    full_text = ""
    doc = fitz.open(path)

    for i, page in enumerate(doc):
        text = page.get_text()
        if len(text.strip()) > 10:
            text = clean_text(text)
        else:
            print(f" Page {i+1} appears to be an image, applying OCR...")
            pix = page.get_pixmap(dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            text = clean_text(text)

        full_text += text + "\n"

    doc.close()
    documents.append(Document(text=full_text, metadata={"file_path": os.path.basename(path)}))

print(f"\n✅ Total documents loaded: {len(documents)}")

# ====== Split into chunks ======
splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents, include_metadata=True)

print(f" Chunking complete. Total chunks created: {len(nodes)}")

# ====== Preview the first few chunks ======
for i, node in enumerate(nodes[:3]):
    print(f"\n Chunk {i+1}")
    print("-" * 50)
    print(f" Source file: {node.metadata.get('file_path', 'N/A')}")
    print(f" Text preview:\n{node.text[:300]}...\n")



 Processing file: law1.pdf

 Processing file: law2.pdf

✅ Total documents loaded: 2
 Chunking complete. Total chunks created: 31

 Chunk 1
--------------------------------------------------
 Source file: law1.pdf
 Text preview:
UNITED STATES DISTRICT COURT FOR THE DISTRICT OF COLUMBIA AMERICAN IMMIGRATION COUNCIL,: et al.,:: Plaintiffs,: Civil Action No.: 23-1952 (RC): v.: Re Document Nos.: 25, 26: EXECUTIVE OFFICE FOR IMMIGRATION: REVIEW,:: Defendant.: MEMORANDUM OPINION GRANTING IN PART AND DENYING IN PART DEFENDANT’S MO...


 Chunk 2
--------------------------------------------------
 Source file: law1.pdf
 Text preview:
As such, the Court grants in part and denies in part each party’s motion for summary judgment and directs the agency to conduct a supplemental search for responsive records. II. FACTUAL BACKGROUND Plaintiffs submitted a FOIA request to EOIR on October 28, 2022, seeking “records of guidelines, proced...


 Chunk 3
--------------------------------------------------
 

In [8]:
# ✅ Create Vector Index and Query Engine
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine()

# ✅ Create base retrievers
base_retriever = index.as_retriever(similarity_top_k=1)

from llama_index.core.retrievers import AutoMergingRetriever
auto_base_retriever = index.as_retriever(similarity_top_k=3)
auto_merging_retriever = AutoMergingRetriever(auto_base_retriever, index.storage_context)

from llama_index.retrievers.bm25 import BM25Retriever
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2, stemmer=Stemmer.Stemmer("english"), language="english")

# Create Hybrid Fusion Retriever
from llama_index.core.retrievers import QueryFusionRetriever

# ✅ Create new bm25 retriever
new_bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5, stemmer=Stemmer.Stemmer("english"), language="english")

# ✅ Create Vector-based retriever
vector_retriever = index.as_retriever(similarity_top_k=5)

# ✅ QueryFusionRetriever
hybrid_retriever = QueryFusionRetriever([new_bm25_retriever, vector_retriever],  retriever_weights=[0.4, 0.6])

DEBUG:bm25s:Building index from IDs objects
DEBUG:bm25s:Building index from IDs objects


In [13]:
from llama_index.core.prompts import PromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine

custom_prompt = PromptTemplate(
    """You are an intelligent and professional legal assistant, helping users by analyzing legal documents retrieved from a trusted knowledge base.

Use the provided case context to answer the user's question. Follow these rules strictly:

---

Answering Rules:
1. If relevant information exists in the context, answer based **only** on that content.
2. If you reference a case, include:
   -  Case Name (if available)
   -  Case Number or URL (e.g., "Case No. 2023-XYZ" or a real hyperlink)
   -  Direct quote or summary from the case as supporting evidence
3. Adapt tone and style:
   - For procedural or factual queries, be concise and formal.
   - For interpretative or advisory questions, provide clarity and explain reasoning briefly.
4. If context is insufficient or irrelevant, **transparently respond with**:
   `" The context does not contain enough information to provide a reliable answer."`
   and suggest what kind of information would be helpful.

---

Context Extracted:
{context_str}

---

User Question:
{query_str}

---

Answer Structure:

Case Reference:
[Case Name] – [Case Number or URL]

Summary / Quote:
"[A short excerpt or reasoning directly supporting the answer]"

Final Answer:
[Your professional and concise answer based on the reference above]

---
"""
)

query_engines = {
    "base": RetrieverQueryEngine.from_args(retriever=base_retriever, text_qa_template=custom_prompt),
    "auto_merge": RetrieverQueryEngine.from_args(retriever=auto_merging_retriever, text_qa_template=custom_prompt),
    "bm25": RetrieverQueryEngine.from_args(retriever=bm25_retriever, text_qa_template=custom_prompt),
    "hybrid": RetrieverQueryEngine.from_args(retriever=hybrid_retriever, text_qa_template=custom_prompt),
}


In [16]:
question = "Is there a limit to how much data I can request under FOIA?"

for name, engine in query_engines.items():
    print(f"\n🔍 [{name.upper()} RETRIEVER]")
    print(engine.query(question))


🔍 [BASE RETRIEVER]
 Case Reference:
American Federation of Government Employees, Local 1924 v. U.S. Department of Homeland Security – Case No. 13-1664 (D.D.C. 2022)

Summary / Quote:
"FOIA still includes some ‘limits aimed at protecting agencies from undue burdens.’"

Final Answer:
There is a limit to how much data you can request under FOIA, as the Act includes some limitations aimed at protecting agencies from undue burdens. The court in American Federation of Government Employees, Local 1924 v. U.S. Department of Homeland Security found that requests that impose an unduly burdensome post-search effort may not be required to be responded to.

🔍 [AUTO_MERGE RETRIEVER]
 Case Reference:
American Federation of Government Employees, Local 1924 v. U.S. Department of Homeland Security, 13 Just., 941 F.3d 563 (D.C. Cir. 2019)

Summary / Quote:
"FOIA still includes some ‘limits aimed at protecting agencies from undue burdens.’"

Final Answer:
While there is no specific limit to the amount of